<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/RLHF%20Reinforcement%20Learning%20from%20Human%20Feedback.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RLHF: Reinforcement Learning from Human Feedback

**Licenc: CC BY-NC-SA 4.0**

Az **RLHF** (Ouyang et al., 2022) az LLM-ek **emberi preferenciákhoz igazítására** szolgál. Ez teszi a ChatGPT-t "chatbot"-tá a "text completion engine" helyett.

## Miért kell RLHF?

A pre-trained LLM-ek:
- ✅ Jól modellezik a nyelvet
- ❌ Nem követik az utasításokat
- ❌ Lehetnek toxikusak, hasztalanok
- ❌ Nem tudják, mit "akar" a felhasználó

Az RLHF **emberi feedback-ből** tanulja meg, mi a "jó" válasz.

## RLHF Pipeline

```
1. Pre-training       → Base LLM (GPT-3)
2. SFT               → Instruction-following (InstructGPT)
3. Reward Model      → Emberi preferenciák tanulása
4. PPO Training      → Policy optimization reward-ra
5. Final Model       → ChatGPT
```

## Tartalomjegyzék
1. Reward Model
2. PPO for LLM
3. RLHF teljes folyamat
4. Constitutional AI és alternatívák

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Reward Model

A **Reward Model** (RM) megtanulja az emberi preferenciákat válasz-párokból.

### Adatgyűjtés

1. Prompt + több válasz generálása
2. **Emberi annotátorok** rangsorolják: A > B > C
3. Párok: (prompt, chosen, rejected)

### Bradley-Terry Model

A preferencia modell valószínűsége:

$$P(y_w \succ y_l | x) = \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))$$

Ahol:
- $r_\theta$: Reward model
- $y_w$: Chosen (winner)
- $y_l$: Rejected (loser)
- $\sigma$: Sigmoid

### Loss

$$\mathcal{L}_{RM} = -\mathbb{E}[\log \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))]$$

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        # Frozen LLM encoder + reward head
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 4, batch_first=True), 2
        )
        self.reward_head = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.encoder(x)
        return self.reward_head(h[:, -1])  # Last token reward

def preference_loss(reward_model, chosen, rejected):
    """
    Bradley-Terry loss: P(chosen > rejected) = σ(r_chosen - r_rejected)
    """
    r_chosen = reward_model(chosen)
    r_rejected = reward_model(rejected)
    return -F.logsigmoid(r_chosen - r_rejected).mean()

rm = RewardModel()
chosen = torch.randn(4, 20, 256)   # Preferred response
rejected = torch.randn(4, 20, 256) # Rejected response
loss = preference_loss(rm, chosen, rejected)
print(f"Preference loss: {loss.item():.4f}")

## 2. PPO for LLM

A **Proximal Policy Optimization (PPO)** optimalizálja az LLM policy-t a reward model alapján.

### PPO Objective

$$\mathcal{L}_{PPO} = \mathbb{E}\left[\min\left(r_t(\theta) \hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

Ahol:
- $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ - probability ratio
- $\hat{A}_t$ - advantage estimate
- $\epsilon$ - clipping parameter (~0.2)

### KL Penalty

A modell ne térjen el túl messze a referencia modelltől:

$$\mathcal{L}_{KL} = \beta \cdot D_{KL}(\pi_\theta || \pi_{ref})$$

### Teljes RLHF objective

$$\mathcal{L} = \mathbb{E}[r_\theta(x, y)] - \beta \cdot D_{KL}(\pi_\theta || \pi_{ref})$$

Maximalizáljuk a reward-ot, de nem térünk el túl messze az SFT modelltől.

In [ ]:
def ppo_objective(log_probs, old_log_probs, advantages, clip_eps=0.2):
    """
    PPO clipped objective.
    """
    ratio = torch.exp(log_probs - old_log_probs)
    clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)
    return -torch.min(ratio * advantages, clipped * advantages).mean()

# KL penalty to stay close to reference model
def kl_penalty(log_probs, ref_log_probs, beta=0.1):
    return beta * (log_probs - ref_log_probs).mean()

# Szimulált értékek
log_probs = torch.randn(100)
old_log_probs = log_probs + torch.randn(100) * 0.1
advantages = torch.randn(100)

loss = ppo_objective(log_probs, old_log_probs, advantages)
print(f"PPO loss: {loss.item():.4f}")

## 3. RLHF teljes folyamat

### Lépések részletesen

| Lépés | Bemenet | Kimenet | Költség |
|-------|---------|---------|---------|
| **Pre-training** | Web corpus | Base LLM | $$$ (nagy compute) |
| **SFT** | (prompt, response) párok | Instruction model | $$ |
| **RM Training** | (prompt, chosen, rejected) | Reward model | $ |
| **PPO** | Prompts + RM | RLHF model | $$ |

### Emberi annotáció

A reward model tanításához **sok emberi annotáció** kell:
- OpenAI: ~40 annotátor, több tízezer összehasonlítás
- Költséges és lassú folyamat!

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

stages = [
    (0.08, 'Pre-trained\nLLM', 'lightblue'),
    (0.25, 'SFT\n(Instructions)', 'lightgreen'),
    (0.42, 'Reward\nModel', 'lightyellow'),
    (0.59, 'PPO\nTraining', 'lightcoral'),
    (0.76, 'RLHF\nModel', 'plum'),
]

for x, text, color in stages:
    ax.add_patch(plt.Rectangle((x, 0.3), 0.14, 0.4, facecolor=color, edgecolor='black', lw=2))
    ax.text(x + 0.07, 0.5, text, ha='center', va='center', fontsize=9)

for i in range(4):
    ax.annotate('', xy=(0.24 + i*0.17, 0.5), xytext=(0.22 + i*0.17, 0.5),
               arrowprops=dict(arrowstyle='->', lw=1.5))

# Human feedback
ax.text(0.49, 0.15, '👤 Human\nPreferences', ha='center', fontsize=8)
ax.annotate('', xy=(0.49, 0.3), xytext=(0.49, 0.22),
           arrowprops=dict(arrowstyle='->', color='green'))

plt.title('RLHF Pipeline', fontsize=12, fontweight='bold', pad=10)
plt.show()

## 4. Constitutional AI és alternatívák

### Constitutional AI (Anthropic)

Az **RLHF alternatívája**: az AI önmagát kritizálja és javítja.

```
1. Generate response
2. AI self-critique: "Ez a válasz sért bármilyen elvet?"
3. Revise based on critique
4. Train on revised responses
```

**Előny**: Kevesebb emberi annotáció kell

### Más alignment módszerek

| Módszer | Leírás |
|---------|--------|
| **DPO** | Direct Preference Optimization (nincs RM) |
| **RLAIF** | RL from AI Feedback |
| **IPO** | Identity Preference Optimization |
| **KTO** | Kahneman-Tversky Optimization |

In [ ]:
print("""
IOAI (Constitutional AI variant):

1. Generate responses
2. AI self-critique (red-teaming)
3. Revise response
4. Human oversight on edge cases
5. Update model
6. Repeat

Előny: Kevesebb emberi label kell, skálázható.
""")

## Összefoglalás

### RLHF komponensek

| Komponens | Cél | Költség |
|-----------|-----|---------|
| **Reward Model** | Emberi preferenciák tanulása | Emberi annotáció |
| **PPO** | Policy optimization reward-ra | Compute |
| **KL penalty** | Ne térjen el túl messze | - |
| **Reference model** | Anchor pont | - |

### Főbb tanulságok

1. **RLHF** = SFT + Reward Model + RL (PPO)
2. A **reward model** emberi összehasonlításokból tanul
3. **KL penalty** megakadályozza a reward hacking-et
4. Költséges: sok emberi annotáció kell

### RLHF kihívások

- **Reward hacking**: A modell "csalni" próbál
- **Costly**: Emberi feedback drága
- **Instabil**: PPO training érzékeny
- **Alignment tax**: Néha rosszabb a raw performance

### Következő lépések

A következő notebookban a **DPO és PPO** összehasonlítását nézzük - a DPO egyszerűbb alternatíva reward model nélkül!